In [1]:
# -*- coding: utf-8 -*-
from typing import Optional, Sequence, List, Tuple
import pandas as pd
from sqlalchemy import create_engine, text
from DATA.stock_invest_function import *

TABLE_NAME = "us_valuation_result"


# --- DB 연결 유틸 ---
def _engine(db_info: dict):
    url = (
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )
    return create_engine(url)

def _latest_created_at(conn, table: str) -> Optional[pd.Timestamp]:
    q = text(f"SELECT MAX(created_at) AS mx FROM {table}")
    s = pd.read_sql(q, conn)['mx'].iloc[0]
    return None if pd.isna(s) else s

# --- 상위 랭킹 메인 함수 ---
def get_top_by_category(
    db_info: dict,
    category: str,                 # 'valuation' 또는 'revenue'
    top_n: int = 20,
    created_at: Optional[str] = None,
    table_name: str = TABLE_NAME,
) -> pd.DataFrame:
    """
    category='valuation'  -> avg_top3 순위(top_n개)
    category='revenue'    -> avg_of_4 순위(top_n개)
    """

    category = category.lower()
    if category not in {"valuation", "revenue"}:
        raise ValueError("category는 'valuation' 또는 'revenue'만 허용됩니다.")

    eng = _engine(db_info)
    with eng.begin() as conn:
        if created_at is None:
            created_at = _latest_created_at(conn, table_name)
            if created_at is None:
                return pd.DataFrame(columns=["ticker", "score"])

        # 1) 우선: 미리 계산된 평균값이 테이블에 있으면 그걸 사용
        target_model = "avg_top3" if category == "valuation" else "avg_of_4"
        q_pre = text(f"""
            SELECT ticker, end_value AS score
              FROM {table_name}
             WHERE category = :cat
               AND model    = :mdl
               AND created_at = :ca
        """)
        pre = pd.read_sql(q_pre, conn, params={"cat": category, "mdl": target_model, "ca": created_at})

        if not pre.empty:
            return pre.sort_values("score", ascending=False).head(top_n).reset_index(drop=True)

        # 2) 없다면: 개별 모델(sarima/lstm/prophet/es)로 계산
        q_raw = text(f"""
            SELECT ticker, model, end_value
              FROM {table_name}
             WHERE category = :cat
               AND model IN ('sarima','lstm','prophet','es')
               AND created_at = :ca
        """)
        df = pd.read_sql(q_raw, conn, params={"cat": category, "ca": created_at})

    if df.empty:
        return pd.DataFrame(columns=["ticker", "score"])

    # 피벗 후 평균 계산
    piv = df.pivot_table(index="ticker", columns="model", values="end_value", aggfunc="last")
    have = [c for c in ["sarima", "lstm", "prophet", "es"] if c in piv.columns]
    if not have:
        return pd.DataFrame(columns=["ticker", "score"])

    if category == "valuation":
        # avg_top3 = 큰 값 3개 평균 (모델이 3개 미만이면 있는 것만 평균)
        def _top3(s: pd.Series) -> float:
            vals = s.dropna().sort_values(ascending=False)
            if len(vals) == 0:
                return float("nan")
            k = min(3, len(vals))
            return float(vals.iloc[:k].mean())
        score = piv[have].apply(_top3, axis=1)
    else:  # category == 'revenue'
        # avg_of_4 = 4개 평균 (결측은 무시; 모델 적으면 있는 것만 평균)
        score = piv[have].mean(axis=1, skipna=True)

    out = (
        score.rename("score")
        .reset_index()
        .sort_values("score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return out


def fetch_all_valuation_results(
    db_info: dict,
    table_name: str = TABLE_NAME,
    columns: Optional[Sequence[str]] = None,   # None이면 *
    category: Optional[str] = None,            # 'valuation' | 'revenue'
    tickers: Optional[Sequence[str]] = None,   # ['AAPL','MSFT'] 등
    created_at: Optional[str] = None,          # 정확 일치 (예: '2025-10-12 15:24:18')
    created_at_like: Optional[str] = None,     # LIKE (예: '2025-10-12%')
    created_ts_like: Optional[str] = None,     # '2025-10-13%' 등
    since: Optional[str] = None,               # start_month_end 하한 (예: '2015-01-01')
    until: Optional[str] = None,               # start_month_end 상한 (예: '2025-12-31')
    order_by: Optional[str] = None,            # 정렬 SQL (None이면 기본 정렬)
    chunksize: int = 100_000,                  # 청크 크기 (메모리 보호)
    to_csv_path: Optional[str] = None,         # 저장 경로 (선택)
    to_excel_path: Optional[str] = None,       # 저장 경로 (선택)
) -> pd.DataFrame:
    """
    us_valuation_result 전체(또는 필터된) 데이터를 DataFrame으로 반환.
    - 대용량 안전: pandas.read_sql(..., chunksize=...)로 청크 결합
    - created_at/created_ts, category, tickers, 날짜 범위 등의 필터 제공
    - 필요 시 CSV/Excel로 바로 저장
    """
    eng = _engine(db_info)

    # SELECT 절
    select_cols = "*"
    if columns:
        select_cols = ", ".join([f"`{c}`" for c in columns])

    # WHERE 절 구성
    where = []
    params = {}

    if category:
        where.append("category = :category")
        params["category"] = category

    if tickers:
        # IN 절 파라미터 바인딩
        tickers = list(tickers)
        tick_params = {f"tk{i}": t for i, t in enumerate(tickers)}
        where.append("ticker IN (" + ", ".join([f":{k}" for k in tick_params.keys()]) + ")")
        params.update(tick_params)

    if created_at:
        where.append("created_at = :created_at")
        params["created_at"] = created_at

    if created_at_like:
        where.append("created_at LIKE :created_at_like")
        params["created_at_like"] = created_at_like

    if created_ts_like:
        where.append("created_ts LIKE :created_ts_like")
        params["created_ts_like"] = created_ts_like

    if since:
        where.append("start_month_end >= :since")
        params["since"] = since

    if until:
        where.append("start_month_end <= :until")
        params["until"] = until

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    # ORDER BY 기본값
    if order_by is None:
        order_by = "ORDER BY created_at DESC, category, model, ticker, start_month_end"

    sql = f"""
        SELECT {select_cols}
          FROM {table_name}
          {where_sql}
          {order_by}
    """

    # 청크로 읽어 결합
    frames: List[pd.DataFrame] = []
    with eng.begin() as conn:
        for chunk in pd.read_sql(text(sql), conn, params=params, chunksize=chunksize):
            frames.append(chunk)
    df = pd.concat(frames, axis=0, ignore_index=True) if frames else pd.DataFrame()

    # 필요 시 저장
    if to_csv_path:
        df.to_csv(to_csv_path, index=False, encoding="utf-8-sig")
    if to_excel_path:
        df.to_excel(to_excel_path, index=False)

    return df

In [2]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}


In [3]:
# 1) 카테고리별 상위표 (가장 최근 배치)

# valuation에서 avg_top3 상위 30개
top_val30 = get_top_by_category(db_info, category="valuation", top_n=20)
print(top_val30)

# revenue에서 avg_of_4 상위 50개 (가장 최근 배치)
top_rev50 = get_top_by_category(db_info, category="revenue", top_n=50)
print(top_rev50)

# 특정 created_at 기준으로 보고 싶으면
top_val = get_top_by_category(db_info, "valuation", 20, created_at="2025-10-12 15:24:18")

  ticker     score
0     ZD  2.073338
  ticker     score
0     ZD  1.491727


In [5]:
# 1) 진짜 전체 다 보기 (기본 정렬, 메모리 안전)
all_df = fetch_all_valuation_results(db_info)


In [8]:
all_df

,id,ticker,category,model,start_month_end,start_value,end_value,growth,created_at,created_ts,updated_ts
0,11790,ZD,revenue,avg_of_4,2025-12-31,1.414592,1.491727,0.054528,2025-12-13 15:56:18,2025-12-14 00:56:18,2025-12-14 00:56:18
1,11789,ZD,revenue,es,2025-12-31,1.410838,1.513396,0.072693,2025-12-13 15:56:18,2025-12-14 00:56:18,2025-12-14 00:56:18
2,11787,ZD,revenue,lstm,2025-12-31,1.395678,1.436776,0.029447,2025-12-13 15:56:18,2025-12-14 00:56:18,2025-12-14 00:56:18
3,11788,ZD,revenue,prophet,2025-12-31,1.444371,1.518130,0.051067,2025-12-13 15:56:18,2025-12-14 00:56:18,2025-12-14 00:56:18
4,11786,ZD,revenue,sarima,2025-12-31,1.407483,1.498604,0.064741,2025-12-13 15:56:18,2025-12-14 00:56:18,2025-12-14 00:56:18
...,...,...,...,...,...,...,...,...,...,...,...
11475,5,AAPL,valuation,avg_top3,2025-11-30,4517.281119,6185.958865,0.369399,2025-11-19 05:33:25,2025-11-19 14:33:26,2025-11-19 14:33:26
11476,4,AAPL,valuation,es,2025-11-30,4773.429346,11510.502593,1.411370,2025-11-19 05:33:25,2025-11-19 14:33:26,2025-11-19 14:33:26
11477,2,AAPL,valuation,lstm,2025-11-30,3297.404945,3616.136597,0.096661,2025-11-19 05:33:25,2025-11-19 14:33:26,2025-11-19 14:33:26
11478,3,AAPL,valuation,prophet,2025-11-30,4345.098009,3431.237405,-0.210320,2025-11-19 05:33:25,2025-11-19 14:33:26,2025-11-19 14:33:26


In [5]:
revenue_forecast = all_df[all_df['model'] == 'avg_of_4'].sort_values('growth', ascending=False)
revenue_forecast['created_at'] = pd.to_datetime(revenue_forecast['created_at'])
revenue_result = revenue_forecast[revenue_forecast['created_at'].dt.date >= pd.to_datetime('2025-10-29').date()].copy()
revenue_result = revenue_result.drop_duplicates(subset=['ticker'], keep='last')


value_forecast = all_df[all_df['model'] == 'avg_top3'].sort_values('growth', ascending=False)
value_forecast['created_at'] = pd.to_datetime(value_forecast['created_at'])
value_result = value_forecast[value_forecast['created_at'].dt.date >= pd.to_datetime('2025-10-29').date()].copy()
value_result = value_result.drop_duplicates(subset=['ticker'], keep='last')

# 1. 각각 growth 기준으로 순위 부여
revenue_result['revenue_rank'] = revenue_result['growth'].rank(ascending=False, method='min')
value_result['value_rank'] = value_result['growth'].rank(ascending=False, method='min')

# 2. ticker 기준으로 병합
combined = pd.merge(
    revenue_result[['ticker', 'growth', 'revenue_rank']],
    value_result[['ticker', 'growth', 'value_rank']],
    on='ticker',
    how='outer',
    suffixes=('_revenue', '_valuation')
)

# 3. 평균 순위 계산
combined['avg_rank'] = combined[['revenue_rank', 'value_rank']].mean(axis=1)

# 4. 최종 순위 부여
combined['final_rank'] = combined['avg_rank'].rank(method='min')

# 5. 최종 순위로 정렬
final_ranking = combined.sort_values('final_rank')

# ===== 깔끔한 출력 (NaN 처리 추가) =====
display_df = final_ranking.copy()
display_df.columns = ['종목', 'Revenue성장률', 'Revenue순위',
                      'Valuation성장률', 'Valuation순위',
                      '평균순위', '최종순위']

# ★★★ NaN 처리 추가 ★★★
# Growth 값 백분율 변환 (NaN은 'N/A'로 표시)
display_df['Revenue성장률'] = display_df['Revenue성장률'].apply(
    lambda x: f"{x*100:.2f}%" if pd.notna(x) else 'N/A'
)
display_df['Valuation성장률'] = display_df['Valuation성장률'].apply(
    lambda x: f"{x*100:.2f}%" if pd.notna(x) else 'N/A'
)

# 순위를 정수로 표시 (NaN은 fillna로 처리)
display_df['Revenue순위'] = display_df['Revenue순위'].fillna(999).astype(int)
display_df['Valuation순위'] = display_df['Valuation순위'].fillna(999).astype(int)
display_df['최종순위'] = display_df['최종순위'].fillna(999).astype(int)
display_df['평균순위'] = display_df['평균순위'].round(1)

# 999는 '-'로 표시
display_df['Revenue순위'] = display_df['Revenue순위'].apply(lambda x: '-' if x == 999 else str(x))
display_df['Valuation순위'] = display_df['Valuation순위'].apply(lambda x: '-' if x == 999 else str(x))
display_df['최종순위'] = display_df['최종순위'].apply(lambda x: '-' if x == 999 else str(x))

# 최종순위로 재정렬
display_df = display_df.sort_values('평균순위')

print("\n" + "=" * 120)
print("최종 랭킹 결과 (Revenue Growth + Valuation Growth)")
print("=" * 120)
print(display_df.to_string(index=False))

# Top 20만 출력
# print("\n" + "=" * 120)
# print("Top 20 추천 종목")
# print("=" * 120)
# top_20 = display_df.head(20)
# print(top_20.to_string(index=False)

# # python# Revenue와 Valuation에 다른 가중치 적용 (예: 6:4)
combined['weighted_rank'] = (combined['revenue_rank'] * 0.6 + combined['value_rank'] * 0.4)
#
combined['final_rank'] = combined['weighted_rank'].rank(method='min')
#
final_ranking = combined.sort_values('final_rank')


최종 랭킹 결과 (Revenue Growth + Valuation Growth)
   종목 Revenue성장률 Revenue순위 Valuation성장률 Valuation순위  평균순위 최종순위
  VIR    132.05%         1      219.07%           1   1.0    1
  RDN     60.63%         3       71.85%           8   5.5    2
 MRNA     66.94%         2       64.61%          10   6.0    3
  WOR     44.62%         6       78.73%           7   6.5    4
 CLSK     30.09%        11       71.12%           9  10.0    5
  ALB     44.30%         7       56.15%          17  12.0    6
 TMDX     44.06%         8       53.41%          23  15.5    7
 INDB     31.17%        10       47.72%          30  20.0    8
  DFH     20.69%        25       55.91%          19  22.0    9
 ASTH     18.54%        30       57.87%          16  23.0   10
 TSLA     16.30%        37       61.59%          12  24.5   11
 KRYS     53.12%         4       40.69%          45  24.5   11
 NAVI     21.58%        23       45.83%          32  27.5   13
 PLTR     17.37%        32       51.15%          25  28.5   14
 CPRX    

In [7]:
display_df[display_df['종목'] == 'INVX']

,종목,Revenue성장률,Revenue순위,Valuation성장률,Valuation순위,평균순위,최종순위


In [32]:
final_ranking['ticker'].unique().tolist()[:150]

['VIR',
 'WOR',
 'RDN',
 'CLSK',
 'MRNA',
 'ASTH',
 'DFH',
 'CPRX',
 'LULU',
 'PLTR',
 'MCHP',
 'GDEN',
 'ADMA',
 'GIII',
 'SMCI',
 'NAVI',
 'INSP',
 'PODD',
 'AXON',
 'AMN',
 'PANW',
 'ACLS',
 'AHCO',
 'UFPT',
 'MGPI',
 'LPG',
 'DXCM',
 'ANET',
 'ULTA',
 'DECK',
 'HRMY',
 'MXL',
 'GNL',
 'AVGO',
 'IAC',
 'SNPS',
 'PLMR',
 'DE',
 'TSLA',
 'DOCN',
 'HCC',
 'AMPH',
 'SEDG',
 'TR',
 'DV',
 'STAA',
 'WDAY',
 'MYRG',
 'ALB',
 'PRVA',
 'ACAD',
 'PLAY',
 'BWA',
 'ACMR',
 'STRL',
 'LW',
 'COLL',
 'GO',
 'DIOD',
 'DGII',
 'ON',
 'PAYO',
 'PRGS',
 'DHI',
 'TKO',
 'PI',
 'FTNT',
 'INSW',
 'FOXA',
 'PCAR',
 'VSAT',
 'VTRS',
 'MU',
 'CPRT',
 'INDB',
 'NXPI',
 'GBX',
 'PLAB',
 'LRN',
 'AMR',
 'COHU',
 'REX',
 'NVR',
 'FRPT',
 'AEO',
 'UBER',
 'DASH',
 'IOSP',
 'AMD',
 'NOW',
 'MODG',
 'CRWD',
 'EG',
 'DDOG',
 'SFBS',
 'MTH',
 'GOOG',
 'NGVT',
 'GOOGL',
 'AMZN',
 'FOX',
 'HLIT',
 'CCS',
 'PHM',
 'DORM',
 'LEN',
 'CVS',
 'RMD',
 'VRTX',
 'CRM',
 'IPG',
 'INCY',
 'TDG',
 'PWR',
 'ROCK',
 'CALX',
 'ALG'

In [7]:
all_date = fetch_table_data(db_info, TABLE_NAME)


✅ 'us_valuation_result' 테이블에서 24010건의 데이터를 가져왔습니다.


,id,ticker,category,model,start_month_end,start_value,end_value,growth,created_at,created_ts,updated_ts
0,1,AAPL,valuation,sarima,2025-10-31,4121.243896,3221.320550,-0.218362,2025-10-11 14:27:48,2025-10-11 23:27:49,2025-10-11 23:27:49
1,2,AAPL,valuation,lstm,2025-10-31,2932.422327,2964.281902,0.010865,2025-10-11 14:27:48,2025-10-11 23:27:49,2025-10-11 23:27:49
2,3,AAPL,valuation,prophet,2025-10-31,4248.317988,3370.664418,-0.206588,2025-10-11 14:27:48,2025-10-11 23:27:49,2025-10-11 23:27:49
3,4,AAPL,valuation,es,2025-10-31,3631.580237,3837.815914,0.056790,2025-10-11 14:27:48,2025-10-11 23:27:49,2025-10-11 23:27:49
4,5,AAPL,valuation,avg_top3,2025-10-31,4000.380707,3476.600294,-0.130933,2025-10-11 14:27:48,2025-10-11 23:27:49,2025-10-11 23:27:49
...,...,...,...,...,...,...,...,...,...,...,...
24005,24226,ZD,revenue,sarima,2025-10-31,1.413773,1.486810,0.051661,2025-10-30 16:02:35,2025-10-31 01:02:34,2025-10-31 01:02:34
24006,24227,ZD,revenue,lstm,2025-10-31,1.393946,1.437900,0.031532,2025-10-30 16:02:35,2025-10-31 01:02:34,2025-10-31 01:02:34
24007,24228,ZD,revenue,prophet,2025-10-31,1.437873,1.439822,0.001355,2025-10-30 16:02:35,2025-10-31 01:02:34,2025-10-31 01:02:34
24008,24229,ZD,revenue,es,2025-10-31,1.421763,1.503708,0.057636,2025-10-30 16:02:35,2025-10-31 01:02:34,2025-10-31 01:02:34
